# 04. 특이값 분해 (SVD)

$$A = U \Sigma V^T$$

처음 봤을 때 기호가 많아서 복잡해 보이는데, 결국 **"모든 행렬을 세 단계 변환으로 쪼개는 것"** 이다.

- $V^T$ : 입력 공간에서 회전
- $\Sigma$ : 각 방향으로 늘리기/줄이기 (특이값이 크기)
- $U$ : 출력 공간에서 회전

**로보틱스 연결:**
- 이미지 압축 → 랭크-$k$ 근사로 데이터 줄이기
- 의사역행렬(pseudoinverse) → 역행렬 없는 시스템 풀기
- ICP 알고리즘 → 포인트 클라우드 정렬에서 최적 회전 $R$ 을 SVD로 구함

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('assets', exist_ok=True)

plt.rcParams['font.family'] = 'Nanum Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. SVD 단계별 시각화 — 단위 원이 어떻게 변하는가

$V^T$ 로 회전 → $\Sigma$ 로 스케일 → $U$ 로 다시 회전.
결과적으로 단위 원은 타원이 된다. 타원의 축 길이 = 특이값.

In [ ]:
A = np.array([[3., 1.], [1., 2.]])
U, S, Vt = np.linalg.svd(A)

# 단위 원
theta = np.linspace(0, 2*np.pi, 300)
circle = np.array([np.cos(theta), np.sin(theta)])

steps = [
    (circle,                          '단위 원 x'),
    (Vt @ circle,                     'Vᵀ x  (회전)'),
    (np.diag(S) @ Vt @ circle,        'Σ Vᵀ x  (스케일)'),
    (U @ np.diag(S) @ Vt @ circle,    'U Σ Vᵀ x = Ax'),
]
colors = ['#534AB7', '#E85D24', '#1D9E75', '#BA7517']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, (pts, title), color in zip(axes, steps, colors):
    ax.plot(pts[0], pts[1], color=color, lw=2.2)
    ax.fill(pts[0], pts[1], alpha=0.12, color=color)
    for vec, ec in [([1,0],'#E85D24'), ([0,1],'#1D9E75')]:
        t_vec = np.array(vec)
        ax.annotate('', xy=t_vec, xytext=[0,0],
                    arrowprops=dict(arrowstyle='->', color=ec, lw=1.8))
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.25)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_title(title, fontsize=11)

plt.suptitle(f'SVD 단계별 변환  |  σ₁={S[0]:.2f}, σ₂={S[1]:.2f}',
             fontsize=13, y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig('assets/04_svd_steps.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'U:\n{U.round(3)}\nΣ: {S.round(3)}\nVt:\n{Vt.round(3)}')
print(f'\n검증 — U @ diag(S) @ Vt:\n{(U @ np.diag(S) @ Vt).round(3)}')

## 2. 이미지 압축 — 랭크-$k$ 근사

$$A \approx A_k = \sum_{i=1}^{k} \sigma_i \vec{u}_i \vec{v}_i^T$$

$k$ 가 작을수록 저장 용량은 줄지만 화질이 떨어진다.
전체 특이값 중 상위 몇 개만으로도 대부분의 정보가 담긴다는 게 핵심.

In [ ]:
np.random.seed(0)
size = 64

# 체커보드 패턴 + 노이즈 (가상 이미지)
img = np.zeros((size, size))
for i in range(8):
    for j in range(8):
        if (i + j) % 2 == 0:
            img[i*8:(i+1)*8, j*8:(j+1)*8] = 1.0
img += 0.15 * np.random.randn(size, size)
img = np.clip(img, 0, 1)

U, S, Vt = np.linalg.svd(img, full_matrices=False)
ranks = [1, 5, 15, 64]

fig, axes = plt.subplots(1, len(ranks)+1, figsize=(16, 4))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('원본', fontsize=11); axes[0].axis('off')

for ax, k in zip(axes[1:], ranks):
    rec = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    err = np.linalg.norm(img - rec, 'fro') / np.linalg.norm(img, 'fro')
    ratio = (2 * k * size) / (size * size) * 100
    ax.imshow(rec, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'rank-{k}\n용량 {ratio:.0f}%\n오차 {err:.3f}', fontsize=10)
    ax.axis('off')

plt.suptitle('SVD 이미지 압축 — 랭크-k 근사', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('assets/04_svd_compression.png', dpi=150, bbox_inches='tight')
plt.show()

# 특이값 누적 기여율
fig2, ax2 = plt.subplots(figsize=(8, 4))
cumvar = np.cumsum(S**2) / np.sum(S**2) * 100
ax2.plot(cumvar, color='#534AB7', lw=2)
ax2.axhline(90, color='#E85D24', lw=1.5, linestyle='--', label='90%')
ax2.axhline(99, color='#1D9E75', lw=1.5, linestyle='--', label='99%')
k90 = np.searchsorted(cumvar, 90) + 1
k99 = np.searchsorted(cumvar, 99) + 1
ax2.axvline(k90, color='#E85D24', lw=1, alpha=0.5)
ax2.axvline(k99, color='#1D9E75', lw=1, alpha=0.5)
ax2.set_xlabel('랭크 k'); ax2.set_ylabel('누적 분산 설명력 (%)')
ax2.set_title(f'특이값 누적 기여율  |  90%→rank {k90}, 99%→rank {k99}', fontsize=11)
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 의사역행렬 (Pseudoinverse) — 역행렬 없을 때

$$A^+ = V \Sigma^+ U^T \quad \text{where} \quad \Sigma^+_{ii} = \begin{cases} 1/\sigma_i & \sigma_i \neq 0 \\ 0 & \sigma_i = 0 \end{cases}$$

로봇 팔 역기구학에서 야코비안 $J$ 가 정방행렬이 아니면 (7DoF 팔 등) 의사역행렬로 관절 속도를 구한다.

$$\dot{q} = J^+ \dot{x}$$

In [ ]:
# 3x2 행렬 — 역행렬 없음, 의사역행렬로 최소제곱 해
A = np.array([[1., 2.], [3., 4.], [5., 6.]])
b = np.array([1., 2., 3.])

U2, S2, Vt2 = np.linalg.svd(A, full_matrices=False)
A_pinv = Vt2.T @ np.diag(1.0 / S2) @ U2.T

x_svd = A_pinv @ b
x_np  = np.linalg.lstsq(A, b, rcond=None)[0]

print('A shape:', A.shape, '→ 역행렬 없음')
print(f'\nSVD 의사역행렬:  x = {x_svd.round(6)}')
print(f'np.linalg.lstsq: x = {x_np.round(6)}')
print(f'\n잔차 ||Ax - b|| = {np.linalg.norm(A @ x_svd - b):.8f}')
print('(이보다 잔차가 작은 해는 없음 — 최소제곱 해)')

# 잔차 vs 다른 해 비교
fig, ax = plt.subplots(figsize=(8, 5))
t = np.linspace(-2, 3, 300)
residuals = [np.linalg.norm(A @ np.array([x_svd[0]+dt, x_svd[1]-dt*2]) - b) for dt in t]
ax.plot(t, residuals, color='#534AB7', lw=2)
ax.axvline(0, color='#E85D24', lw=2, linestyle='--', label=f'최소제곱 해 (잔차={np.linalg.norm(A@x_svd-b):.4f})')
ax.set_xlabel('x[0] 편차'); ax.set_ylabel('잔차 크기')
ax.set_title('의사역행렬 해가 잔차를 최소화함', fontsize=11)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 요약

| 개념 | 수식 | 로보틱스 활용 |
|------|------|---------------|
| SVD 분해 | $A = U\Sigma V^T$ | 모든 행렬에 적용 가능 |
| 랭크-k 근사 | $A_k = \sum_{i=1}^k \sigma_i u_i v_i^T$ | 이미지/데이터 압축 |
| 의사역행렬 | $A^+ = V\Sigma^+ U^T$ | 역기구학 (야코비안) |
| ICP 회전 | SVD of $H = X Y^T$ | 포인트 클라우드 정렬 |

**다음 노트북:** `05_3d_rotations.ipynb` — 회전행렬·오일러각·쿼터니언